In [ ]:
import math
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from utils import load_video, play_video, save_video
from IPython.display import clear_output

In [ ]:
video = load_video("data/pnp/flight4.h264", gray=True)

In [ ]:
calib_data = np.load("data/calib/calib_data1.npz")
calib_mtx = calib_data["mtx"]
calib_dist = calib_data["dist"]
calib_new_mtx = calib_data["new_mtx"]
print(calib_mtx)

In [ ]:
cb_rows, cb_cols = 3, 5
cb_square_size = 0.070

objp = np.empty((cb_rows * cb_cols, 3), np.float32)
for i in range(cb_rows):
    for j in range(cb_cols):
        objp[i * cb_cols + j] = [j * cb_square_size, i * cb_square_size, 0]

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

In [ ]:
# x stays the same, y becomes z, z becomes -y
DRONE_ROT = np.array([[1, 0, 0], [0, 0, 1], [0, -1, 0]])

def get_cam_pos(img, objp, cb_size, calib_mtx, calib_dist, criteria):
    ret, corners = cv2.findChessboardCorners(
        img, cb_size, None, cv2.CALIB_CB_FAST_CHECK
    )
    if ret:
        corners_sb = cv2.cornerSubPix(img, corners, (11, 11), (-1, -1), criteria)
        if corners_sb[0][0][0] > corners_sb[-1][0][0]:
            corners_sb = corners_sb[::-1]

        _, rvec, tvec = cv2.solvePnP(
            objp, corners_sb, calib_mtx, calib_dist, flags=cv2.SOLVEPNP_IPPE
        )

        R, _ = cv2.Rodrigues(rvec)
        cam_pos = DRONE_ROT @ (-R.T @ tvec)
        return cam_pos.flatten()
    return None

In [ ]:
raw_positions = []
for frame in tqdm(video):
    pos = get_cam_pos(
        frame,
        objp,
        (cb_cols, cb_rows),
        calib_mtx,
        calib_dist,
        criteria,
    )
    if pos is None:
        raw_positions.append([np.nan, np.nan, np.nan])
    else:
        raw_positions.append(pos)
raw_positions = np.array(raw_positions)

In [ ]:
# plot
fig, ax = plt.subplots(3, 1, figsize=(10, 8))
labels = ["X", "Y", "Z"]
for i in range(3):
    ax[i].plot(raw_positions[:, i])
    ax[i].set_ylabel(f"{labels[i]} (m)")
    ax[i].set_xlabel("Frame")
    ax[i].grid()
plt.tight_layout()
plt.show()

In [ ]:
class KalmanFilter:
    def __init__(self, process_noise=0.01, measurement_noise=0.5):
        self.pos = None
        self.vel = None
        self.P = np.eye(3) * 1.0  # covariance
        self.Q = np.eye(3) * process_noise  # process noise
        # self.R = np.eye(3) * measurement_noise  # measurement noise
        self.R = np.diag([measurement_noise, measurement_noise, measurement_noise])

    def update(self, measurement, dt=1 / 30):
        if measurement is None:
            # Predict only
            if self.pos is not None and self.vel is not None:
                self.pos = self.pos + self.vel * dt
                self.P = self.P + self.Q
            return self.pos

        measurement = np.array(measurement)

        if self.pos is None:
            self.pos = measurement
            self.vel = np.zeros(3)
            return self.pos

        # Predict
        predicted_pos = self.pos + self.vel * dt
        self.P = self.P + self.Q

        # Update
        K = self.P @ np.linalg.inv(self.P + self.R)  # Kalman gain
        innovation = measurement - predicted_pos

        self.pos = predicted_pos + K @ innovation
        self.vel = (
            self.vel + (K @ innovation) / dt * 0.1
        )  # slowly update velocity estimate
        self.P = (np.eye(3) - K) @ self.P

        return self.pos

In [ ]:
kf = KalmanFilter(process_noise=0.01, measurement_noise=0.5)
filtered_positions = []
for pos in raw_positions:
    if np.any(np.isnan(pos)):
        filtered_pos = kf.update(None)
    else:
        filtered_pos = kf.update(pos)
    filtered_positions.append(filtered_pos)
filtered_positions = np.array(filtered_positions)

In [ ]:
# plot
fig, ax = plt.subplots(3, 1, figsize=(10, 8))
labels = ["X", "Y", "Z"]
for i in range(3):
    ax[i].plot(filtered_positions[:, i])
    ax[i].set_ylabel(f"{labels[i]} (m)")
    ax[i].set_xlabel("Frame")
    ax[i].grid()
plt.tight_layout()
plt.show()

In [ ]:
# load pos_filter_output.npz and compare
pf_data = np.load("data/flight/pf_output1.npz")
times = pf_data["times"]
pf_positions = pf_data["poses"]

In [ ]:
# plot
start = times[0]
times = times - start
fig, ax = plt.subplots(3, 1, figsize=(10, 8))
labels = ["X", "Y", "Z"]
for i in range(3):
    ax[i].plot(times, pf_positions[:, i])
    ax[i].set_ylabel(f"{labels[i]} (m)")
    ax[i].set_xlabel("Time (s)")
    ax[i].grid()
    ax[i].set_ylim(-1, 1)
    ax[i].legend()
plt.tight_layout()

In [ ]:
%matplotlib widget

# load pf_output_final.npz and plot in 3d using matplotlib
pf_final_data = np.load("data/pf_output_final.npz")
pf_final_positions = pf_final_data["poses"]

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(
    pf_final_positions[:, 0],
    pf_final_positions[:, 1],
    pf_final_positions[:, 2],
    label="Position",
)
# Add green point at the start
ax.scatter(
    pf_final_positions[0, 0],
    pf_final_positions[0, 1],
    pf_final_positions[0, 2],
    color="green",
    s=60,
    label="Start"
)
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.set_title("3D Position Plot")
ax.legend()
plt.show()